In [1]:
import numpy as np
import os
import time
import scipy.sparse as sp

## Задача 6

6.1

In [2]:
arr = np.arange(20)
print(arr[1::3])
print(arr[14:4:-1])
print(arr[:-6:-1])

[ 1  4  7 10 13 16 19]
[14 13 12 11 10  9  8  7  6  5]
[19 18 17 16 15]


6.2

In [3]:
M = np.random.randint(0, 10, size=(6, 6))
print(M)
print(M[2:5, 1:4])
print(np.diagonal(M))
i, j = np.indices(M.shape)
print(M[(i + j) % 2 == 0])

[[9 9 1 9 1 0]
 [0 6 1 5 7 5]
 [7 9 3 4 5 0]
 [9 8 2 3 5 1]
 [5 3 9 3 9 8]
 [1 8 0 2 0 0]]
[[9 3 4]
 [8 2 3]
 [3 9 3]]
[9 6 3 3 9 0]
[9 1 1 6 5 5 7 3 5 8 3 1 5 9 9 8 2 0]


6.3

In [4]:
a = np.array([10, 20, 30, 40, 50, 60])
m = np.arange(12).reshape(3, 4)
print(a[[0, 2, 4]])
print(m[[0, 1, 2], [1, 3, 0]])
print(m[np.ix_([0, 2], [1, 3])])

[10 30 50]
[1 7 8]
[[ 1  3]
 [ 9 11]]


6.4

In [5]:
x = np.random.randn(1000)
print((x > 1).sum())

155


In [6]:
x_nonneg = x.copy()
x_nonneg[x_nonneg < 0] = 0
print((x_nonneg < 0).sum())

0


In [7]:
mask = np.zeros(len(x), dtype=bool)
mask[1:-1] = np.sign(x[:-2]) != np.sign(x[2:])
x_zeroed = x.copy()
x_zeroed[mask] = 0
print(mask.sum())

471


## Задача 7

7.1

In [8]:
shape = (50000, 200)
big = np.memmap("big.dat", dtype=np.float32, mode="w+", shape=shape)
for start in range(0, shape[0], 10000):
    big[start:start + 10000] = np.random.rand(10000, shape[1]).astype(np.float32)
big.flush()
del big

print(os.path.getsize("big.dat"))
print(50000 * 200 * 4)
print(os.path.getsize("big.dat") == 50000 * 200 * 4)

40000000
40000000
True


7.2

In [9]:
big = np.memmap("big.dat", dtype=np.float32, mode="r", shape=shape)
batch = 5000
g_min = np.inf
g_max = -np.inf
total = 0.0
for start in range(0, shape[0], batch):
    part = big[start:start + batch]
    g_min = min(g_min, part.min())
    g_max = max(g_max, part.max())
    total += part.sum(dtype=np.float64)
mean = total / (shape[0] * shape[1])
print(g_min, g_max, mean)

6.4754064e-09 0.9999999 0.4999694865135706


7.3

In [10]:
n = 3000
a_c = np.random.rand(n, n)
a_f = np.asfortranarray(a_c)

def sum_rows(a):
    s = 0.0
    for i in range(n):
        for j in range(n):
            s += a[i, j]
    return s

def sum_cols(a):
    s = 0.0
    for j in range(n):
        for i in range(n):
            s += a[i, j]
    return s

In [11]:
%time sum_rows(a_c)

CPU times: total: 1.05 s
Wall time: 1.08 s


np.float64(4499756.658009867)

In [12]:
%time sum_cols(a_c)

CPU times: total: 1.09 s
Wall time: 1.16 s


np.float64(4499756.658010215)

In [13]:
%time sum_rows(a_f)

CPU times: total: 1 s
Wall time: 1.07 s


np.float64(4499756.658009867)

In [14]:
%time sum_cols(a_f)

CPU times: total: 922 ms
Wall time: 945 ms


np.float64(4499756.658010215)

7.4

In [15]:
N = 10**7
a = np.random.rand(N)
b = np.random.rand(N)
la = a.tolist()
lb = b.tolist()

def calc_loop(la, lb):
    res = []
    for x, y in zip(la, lb):
        res.append((x + y) / (x - y + 1))
    return res

def calc_np(a, b):
    return (a + b) / (a - b + 1)

out = np.empty_like(a)
tmp = np.empty_like(a)

def calc_out(a, b, out, tmp):
    np.add(a, b, out=out)
    np.subtract(a, b, out=tmp)
    np.add(tmp, 1, out=tmp)
    np.divide(out, tmp, out=out)
    return out

In [16]:
%time r1 = calc_loop(la, lb)

CPU times: total: 922 ms
Wall time: 965 ms


In [17]:
%timeit calc_np(a, b)

86.6 ms ± 2.82 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [18]:
%timeit calc_out(a, b, out, tmp)

34.4 ms ± 476 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [19]:
print(np.allclose(r1, calc_np(a, b)))
print(np.allclose(calc_np(a, b), calc_out(a, b, out, tmp)))

True
True


## Задача 8

8.1

In [20]:
n = 5000
row = np.random.randint(0, n, 10000).tolist()
col = np.random.randint(0, n, 10000).tolist()
data = (np.random.rand(10000) + 1).tolist()

A_coo = sp.coo_matrix((data, (row, col)), shape=(n, n))
A_csr = A_coo.tocsr()

unique_pairs = len(set(zip(row, col)))
print("записей в COO:", A_coo.nnz)
print("уникальных пар (row, col):", unique_pairs)
print("nnz в CSR:", A_csr.nnz)
print("nnz корректен:", A_csr.nnz == unique_pairs)
print("сумма значений сохранилась:", np.isclose(A_coo.sum(), A_csr.sum()))

записей в COO: 10000
уникальных пар (row, col): 10000
nnz в CSR: 10000
nnz корректен: True
сумма значений сохранилась: True


8.2

In [21]:
n = 4000
S = sp.random(n, n, density=0.005, format="csr")
D = S.toarray()
x = np.random.rand(n)

dense_bytes = D.nbytes
csr_bytes = S.data.nbytes + S.indices.nbytes + S.indptr.nbytes
print("dense, МБ:", dense_bytes / 1024**2)
print("CSR, МБ:  ", csr_bytes / 1024**2)
print("разница в", dense_bytes / csr_bytes, "раз")

dense, МБ: 122.0703125
CSR, МБ:   0.9307899475097656
разница в 131.1470034958873 раз


In [22]:
%timeit D @ x

4.11 ms ± 184 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [23]:
%timeit S @ x

61.5 μs ± 1.67 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


8.3

In [24]:
import timeit

n = 2000
x = np.random.rand(n)
densities = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1.0]

print(f"{'density':>8} {'dense, мс':>10} {'sparse, мс':>11}")
point = None
for d in densities:
    S = sp.random(n, n, density=d, format="csr")
    D = S.toarray()
    t_dense = min(timeit.repeat(lambda: D @ x, number=20, repeat=5)) / 20
    t_sparse = min(timeit.repeat(lambda: S @ x, number=20, repeat=5)) / 20
    print(f"{d:8} {t_dense*1e3:10.3f} {t_sparse*1e3:11.3f}")
    if point is None and t_sparse >= t_dense:
        point = d

print("sparse перестаёт выигрывать при плотности:", point)

 density  dense, мс  sparse, мс
   0.001      0.744       0.006
   0.005      0.780       0.018
    0.01      0.687       0.030
    0.05      0.688       0.086
     0.1      0.672       0.176
     0.2      0.667       0.406
     0.5      0.750       1.351
     1.0      0.689       2.514
sparse перестаёт выигрывать при плотности: 0.5
